# Final CIFAR-10 evaluation: RGB + AVG + NTSC flip fusion

This notebook evaluates the completed horizontal-flip CNN-SVM experiment exactly once against the official CIFAR-10 test split. Model selection is frozen from the completed search ledger before test images or labels are loaded.

The current switches are intentionally disabled. Review the ledger verification result before enabling an expensive stage.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'flip_features.py').is_file():
    ROOT = ROOT / 'tugas-7'
if not (ROOT / 'flip_features.py').is_file():
    raise FileNotFoundError('Run from the repository root or tugas-7')
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT))

from final_flip_testing import (
    EXPECTED_SELECTION, FINAL_DIRECTORY, evaluate_official,
    create_final_directory, extract_official_test_features,
    fit_final_model, load_development_features, load_frozen_selection,
    save_final_artifacts,
)

RUN_FINAL_MODEL_FIT = True
RUN_TEST_FEATURE_EXTRACTION = True
RUN_OFFICIAL_EVALUATION = True
SAVE_FINAL_ARTIFACTS = True

print('ROOT:', ROOT)
print('All expensive stages disabled:', not any((RUN_FINAL_MODEL_FIT, RUN_TEST_FEATURE_EXTRACTION, RUN_OFFICIAL_EVALUATION, SAVE_FINAL_ARTIFACTS)))

ROOT: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-7
All expensive stages disabled: False


## 1. Freeze and verify model selection

The ledger must contain exactly 45 completed candidates and its ranked winner must match the reviewed parameters below. A mismatch stops the notebook before official test access.

In [2]:
selection = load_frozen_selection(ROOT)
assert selection['parameters'] == EXPECTED_SELECTION
print('Frozen selection:', selection)

Frozen selection: {'ledger_path': '/home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-7/feature-engineering/experiments/fusion_flip_v1_001/svm_search/search_state.json', 'ledger_sha256': 'b45667713649e34d077d128a7b0a7faa87dacef80dc168ee325390d6003719fd', 'completed_candidates': 45, 'validation_accuracy': 0.9515, 'validation_correct': 9515, 'parameters': {'C': 1.0, 'gamma': 0.00015811388300841897, 'weights': [1.0, 0.5, 0.5]}, 'kernel': 'rbf'}


In [3]:
import json
import numpy as np

from flip_features import MODELS, combined_path
from fusion_search import file_digest

with np.load(
    combined_path(ROOT, "fusion_flip_v1_001"),
    allow_pickle=False
) as archive:
    metadata = json.loads(
        archive["provenance_json"].item()
    )

for representation, model_path in MODELS.items():
    expected_hash = metadata["sources"][representation]["cnn_sha256"]
    actual_hash = file_digest(ROOT / model_path)

    assert expected_hash == actual_hash, (
        f"{representation} checkpoint changed!"
    )

    print(f"{representation}: checkpoint verified")

print("All development/test checkpoint identities match.")

RGB: checkpoint verified
AVG: checkpoint verified
NTSC: checkpoint verified
All development/test checkpoint identities match.


## 2. Fit the final SVM on all 50,000 development images

The existing flip-averaged train and validation archives are concatenated in their stored order. `VarianceThreshold(0)` and `StandardScaler` are fitted again on these 50,000 rows, then transformed arrays are cast to `float32` and weighted by RGB, AVG, NTSC blocks.

In [4]:
development = None
final_model = None
selector = None
scaler = None
block_ids = None
fit_provenance = None
if RUN_FINAL_MODEL_FIT:
    X_development, y_development, development_provenance = load_development_features(ROOT)
    final_model, selector, scaler, block_ids, fit_provenance = fit_final_model(
        X_development, y_development, selection
    )
    development = (X_development, y_development)
    print('Final SVM fitted on:', X_development.shape)
else:
    print('Final model fitting disabled.')

Final SVM fitted on: (50000, 1536)


## 3. Extract official test features

After selection is frozen, the official test images and labels are loaded. RGB, AVG, and NTSC checkpoints are loaded sequentially; each block is `(original + horizontally_flipped) / 2` from the 512-dimensional `svm_features` layer with batch size 128.

In [5]:
test_features = None
test_labels = None
extraction_provenance = None
if RUN_TEST_FEATURE_EXTRACTION:
    test_features, test_labels, extraction_provenance = extract_official_test_features(ROOT)
    assert test_features.shape == (10000, 1536)
    assert test_features.dtype == 'float32'
    print('Official test features:', test_features.shape)
else:
    print('Official test feature extraction disabled.')

I0000 00:00:1791428770.349105   64058 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791428770.384881   64058 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791428771.373214   64058 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791428773.573736   64058 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3672 MB mem

 5/79 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step

I0000 00:00:1791428780.356595   64179 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


79/79 ━━━━━━━━━━━━━━━━━━━━ 11s 70ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step
RGB: extracted 10000 rows x 512 features
79/79 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step
AVG: extracted 10000 rows x 512 features
79/79 ━━━━━━━━━━━━━━━━━━━━ 5s 44ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step
NTSC: extracted 10000 rows x 512 features
Official test features: (10000, 1536)


## 4. Official evaluation

Test labels are used only here, for the final accuracy and diagnostic report. No test result changes model selection or preprocessing.

In [6]:
evaluation = None
if RUN_OFFICIAL_EVALUATION:
    if final_model is None or test_features is None:
        raise RuntimeError('Enable final fitting and test feature extraction first')
    evaluation = evaluate_official(
        final_model, test_features, test_labels, selector, scaler, block_ids,
        selection['parameters']['weights'],
    )
    print(f"Accuracy: {evaluation['accuracy']:.4%} ({evaluation['correct']}/{evaluation['rows']})")
    print('Strictly greater than 95%:', evaluation['strictly_greater_than_95_percent'])
    print('Confusion matrix:')
    display(evaluation['confusion_matrix'])
    print('Classification report:')
    display(evaluation['classification_report'])
else:
    print('Official evaluation disabled.')

Accuracy: 94.1700% (9417/10000)
Strictly greater than 95%: False
Confusion matrix:


[[951, 2, 13, 4, 1, 0, 2, 2, 19, 6],
 [0, 987, 0, 1, 0, 0, 0, 0, 1, 11],
 [9, 0, 928, 14, 16, 8, 16, 4, 2, 3],
 [3, 1, 20, 864, 18, 60, 24, 4, 3, 3],
 [1, 1, 13, 13, 947, 5, 9, 10, 1, 0],
 [4, 1, 13, 65, 15, 889, 6, 6, 0, 1],
 [3, 0, 8, 12, 4, 2, 967, 2, 0, 2],
 [6, 0, 5, 9, 13, 8, 2, 955, 2, 0],
 [13, 6, 3, 2, 0, 0, 0, 0, 968, 8],
 [3, 28, 1, 2, 0, 0, 0, 0, 5, 961]]

Classification report:


{'0': {'precision': 0.9577039274924471,
  'recall': 0.951,
  'f1-score': 0.9543401906673357,
  'support': 1000.0},
 '1': {'precision': 0.9619883040935673,
  'recall': 0.987,
  'f1-score': 0.9743336623889437,
  'support': 1000.0},
 '2': {'precision': 0.9243027888446215,
  'recall': 0.928,
  'f1-score': 0.9261477045908184,
  'support': 1000.0},
 '3': {'precision': 0.8762677484787018,
  'recall': 0.864,
  'f1-score': 0.8700906344410876,
  'support': 1000.0},
 '4': {'precision': 0.9339250493096647,
  'recall': 0.947,
  'f1-score': 0.9404170804369414,
  'support': 1000.0},
 '5': {'precision': 0.9146090534979424,
  'recall': 0.889,
  'f1-score': 0.9016227180527383,
  'support': 1000.0},
 '6': {'precision': 0.942495126705653,
  'recall': 0.967,
  'f1-score': 0.9545903257650543,
  'support': 1000.0},
 '7': {'precision': 0.9715157680569685,
  'recall': 0.955,
  'f1-score': 0.9631870902672718,
  'support': 1000.0},
 '8': {'precision': 0.967032967032967,
  'recall': 0.968,
  'f1-score': 0.9675162

## 5. Persist the final result

Artifacts are written to a new dedicated directory only. Existing completed output is never overwritten. Provenance includes the ledger, development archive, checkpoint hashes, feature ordering, and evaluation hashes.

In [7]:
if SAVE_FINAL_ARTIFACTS:
    if evaluation is None or final_model is None or development is None:
        raise RuntimeError('Run final fitting and official evaluation before saving')
    output_directory = create_final_directory(ROOT, FINAL_DIRECTORY)
    provenance = {
        'selection': selection,
        'development': development_provenance,
        'fit': fit_provenance,
        'test_extraction': extraction_provenance,
        'preprocessing': 'VarianceThreshold(0), StandardScaler, float32, then block weights',
        'feature_order': ['RGB', 'AVG', 'NTSC'],
        'feature_dims': [512, 512, 512],
    }
    print(save_final_artifacts(
        output_directory, final_model, selector, scaler, selection['parameters'],
        provenance, evaluation,
    ))
else:
    print('Artifact persistence disabled.')

{'final_svm.joblib': '1a36f7c60779585f1dd899fcbbc22519229c836d7af10778283fcc1ef9b792d5', 'preprocessing.joblib': 'a9374edbed8865a7ae4874b3f9db5bb92c354a5b10ff5f93fc69e858cfc05c97', 'parameters.json': '517650e62fb3ff4ddb73a62aef0099a97ee3adc6f98ce3194dc29ebbd6757986', 'provenance.json': '42bb005b05683cec78db09b8b1d6f52cbbd9cd8ce8815addc294c0128bea6f03', 'evaluation.json': 'd5edf4b505eaee439102fb6c88e00811106c8d01f90558152da597943303dcef'}


In [8]:
from sklearn.svm import SVC
import numpy as np

from flip_features import combined_path, verify_combined
from final_flip_testing import (
    fit_preprocessing,
    transform_features,
    load_frozen_selection,
)

replay_data, _ = verify_combined(
    combined_path(ROOT, "fusion_flip_v1_001"),
    "fusion_flip_v1_001"
)

replay_selection = load_frozen_selection(ROOT)
params = replay_selection["parameters"]

replay_selector, replay_scaler, replay_blocks, _ = fit_preprocessing(
    replay_data["X_train_combined"],
    expected_rows=40000
)

replay_train = transform_features(
    replay_data["X_train_combined"],
    replay_selector, replay_scaler, replay_blocks,
    params["weights"]
)

replay_val = transform_features(
    replay_data["X_val_combined"],
    replay_selector, replay_scaler, replay_blocks,
    params["weights"]
)

replay_model = SVC(
    kernel="rbf",
    C=params["C"],
    gamma=params["gamma"],
    cache_size=2048,
)

replay_model.fit(replay_train, replay_data["y_train"])

replay_predictions = replay_model.predict(replay_val)
replay_correct = int(np.sum(
    replay_predictions == replay_data["y_val"]
))

print(f"Reproduced validation: {replay_correct / 10000:.2%}")
print(f"Correct predictions: {replay_correct}/10000")
print(f"Matches ledger: {replay_correct == 9515}")

Reproduced validation: 95.15%
Correct predictions: 9515/10000
Matches ledger: True
